Notebook 4 — LSTM Regressor (Pronóstico de Precios).

CELDA 1 — Instalación

In [1]:
!pip install yfinance tensorflow scikit-learn statsmodels --quiet

CELDA 2 — Importaciones

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import json

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")

CELDA 3 — Configuración

In [13]:
TICKER = "BVN"
VENTANA = 60
HORIZONTE = 30

df = yf.download(
    TICKER,
    period="3y",
    progress=False,
    auto_adjust=True
)

df = df.reset_index()
df = df.dropna()

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

precios = df["Close"].values.reshape(-1,1)

print(df.head())

Price       Date     Close      High       Low      Open   Volume
0     2023-06-20  7.103264  7.150682  6.942042  7.055846  1569300
1     2023-06-21  7.122231  7.179133  6.923075  7.112748  1733900
2     2023-06-22  7.093781  7.131715  6.970493  6.970493  1288800
3     2023-06-23  7.055846  7.292938  7.046362  7.169650  1720000
4     2023-06-26  7.122231  7.198101  7.084296  7.131715  1587800


CELDA 4 — Escalamiento

In [14]:
scaler = MinMaxScaler()

datos_scaled = scaler.fit_transform(precios)

X = []
y = []

for i in range(VENTANA, len(datos_scaled)):
    X.append(datos_scaled[i-VENTANA:i])
    y.append(datos_scaled[i])

X = np.array(X)
y = np.array(y)

print(X.shape)

(692, 60, 1)


CELDA 5 — Train/Test

In [15]:
split = int(len(X)*0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(553, 60, 1)
(139, 60, 1)


CELDA 6 — Modelo LSTM

In [16]:
model = Sequential()

model.add(
    LSTM(
        64,
        return_sequences=True,
        input_shape=(VENTANA,1)
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(32)
)

model.add(Dropout(0.2))

model.add(Dense(16, activation="relu"))

model.add(Dense(1))

model.compile(
    optimizer="adam",
    loss="mse"
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 80ms/step - loss: 0.0190 - val_loss: 0.0025
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 0.0032 - val_loss: 0.0132
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 0.0018 - val_loss: 0.0099
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 0.0015 - val_loss: 0.0038
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 0.0011 - val_loss: 0.0030
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 98ms/step - loss: 0.0011 - val_loss: 0.0040
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 9.9235e-04 - val_loss: 0.0033
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - loss: 8.7910e-04 - val_loss: 0.0029
Epoch 9/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - loss: 8.9446e-04 - val_loss: 0.0026
Epoch 10/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 8.1974e-04 - val_loss: 0.0028
Epoch 11/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 7.9527e-04 - val_loss: 0.0026
Epoch 12/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 5

CELDA 7 — Predicciones históricas

In [17]:
pred = model.predict(X_test)

pred_real = scaler.inverse_transform(pred)
y_real = scaler.inverse_transform(y_test)

mask = ~np.isnan(y_real.flatten())

y_real = y_real[mask]
pred_real = pred_real[mask]

rmse = np.sqrt(
    mean_squared_error(y_real, pred_real)
)

mae = mean_absolute_error(
    y_real,
    pred_real
)

r2 = r2_score(
    y_real,
    pred_real
)

rmse_pct = rmse / np.mean(y_real) * 100

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step
RMSE: 3.8598580668188074
MAE: 3.002832687158379
R2: 0.09298037635853784


In [18]:
pred_hist_scaled = model.predict(X, verbose=0)
pred_hist_real = scaler.inverse_transform(pred_hist_scaled)

predHist = [None] * VENTANA

for valor in pred_hist_real:
    predHist.append(float(valor[0]))

CELDA 8 — Pronóstico futuro (30 días)

In [19]:
ultimo = datos_scaled[-VENTANA:]

entrada = ultimo.reshape(1, VENTANA, 1)

futuro = []

for i in range(HORIZONTE):

    p = model.predict(entrada, verbose=0)

    futuro.append(p[0][0])

    nueva = np.append(
        entrada[0][1:],
        p
    )

    entrada = nueva.reshape(1, VENTANA, 1)

pred_fut = scaler.inverse_transform(
    np.array(futuro).reshape(-1,1)
)

pred_fut = pred_fut.flatten()

# Fechas futuras

ultima_fecha = pd.to_datetime(df["Date"].iloc[-1])

fechasFut = []

while len(fechasFut) < HORIZONTE:
    ultima_fecha += pd.Timedelta(days=1)

    if ultima_fecha.weekday() < 5:
        fechasFut.append(
            ultima_fecha.strftime("%Y-%m-%d")
        )

CELDA 9 — ARIMA

In [20]:
serie = df["Close"]

arima = ARIMA(
    serie,
    order=(2,1,2)
)

modelo_arima = arima.fit()

pred_arima = modelo_arima.forecast(HORIZONTE)

pred_arima = pred_arima.values

CELDA 10 — Bandas de confianza

In [21]:
desv = np.std(y_real - pred_real)

banda_sup = pred_fut + 1.96*desv
banda_inf = pred_fut - 1.96*desv

CELDA 11 — Curvas de loss

In [22]:
lossHist = history.history["loss"]
valLossHist = history.history["val_loss"]

In [23]:
fechasHist = (
    pd.to_datetime(df["Date"])
    .dt.strftime("%Y-%m-%d")
    .tolist()
)

preciosReales = (
    df["Close"]
    .astype(float)
    .tolist()
)

CELDA 12 — Exportación JSON

In [24]:
salida = {

    "fechasHist": fechasHist,

    "preciosReales": preciosReales,

    "predHist": predHist,

    "fechasFut": fechasFut,

    "metricas": {
        "rmseUSD": round(float(rmse),4),
        "rmsePct": round(float(rmse_pct),2),
        "mae": round(float(mae),4),
        "r2": round(float(r2),4)
    },

    "predFut": pred_fut.tolist(),

    "predARIMA": pred_arima.tolist(),

    "bandaSup": banda_sup.tolist(),

    "bandaInf": banda_inf.tolist(),

    "lossHist": [
        float(x)
        for x in lossHist
    ],

    "valLossHist": [
        float(x)
        for x in valLossHist
    ]
}

with open(
    "lstm_regressor.json",
    "w"
) as f:
    json.dump(
        salida,
        f,
        indent=4
    )

print("JSON exportado.")

JSON exportado.


In [25]:
from google.colab import files

files.download("lstm_regressor.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>